<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/homework/WRIT20833_HW3_2026_ANSWER_KEY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3 — Sentiment Analysis — INSTRUCTOR ANSWER KEY

**WRIT 20833 — Intro to Coding in the Humanities**

> Solutions are one valid approach; students may reach the same result differently. Teaching
> notes appear under each solution. **Ungrading:** this key is a **worked example + discussion
> fodder, not a rubric** — reward engagement and honest human-vs-VADER judgment over "correct"
> labels. It **runs top-to-bottom on the course corpus** (the BYO fallback, since student data
> can't be keyed). Counts reflect that corpus: **123 rows, 2026-06**. *(Note: the corpus is 123
> one-per-row lines — a few are short CSV→txt wrap-fragments; the `data/README.md` count has been
> corrected from an earlier "93".)*

---

In [ ]:
# SETUP — run this cell first. (You're not expected to read every line yet — that's fine.)
import re, os, urllib.request
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

# VADER is a ready-made sentiment tool. It is not built into Python, so we install it,
# then make one analyzer we reuse. (More on "borrowing" tools in the note after A5.)
!pip install -q vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

# Same word-splitter + stopwords you built in HW2 (we reuse our own tools).
def split_into_words(any_chunk_of_text):
    return re.split(r"\W+", str(any_chunk_of_text).lower())
stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now", "ve", "ll", "amp"]

def load_course_comments():
    """Load the course YouTube-comments corpus into a DataFrame, one comment per row."""
    filename = "tc_youtube_comments.txt"
    text = None
    for path in (filename, os.path.join("notebooks", "data", filename),
                 os.path.join("data", filename), os.path.join("..", "data", filename)):
        if os.path.exists(path):
            text = open(path, encoding="utf-8").read(); break
    if text is None:
        url = "https://raw.githubusercontent.com/TCU-DCDA/WRIT20833_2026/main/notebooks/data/" + filename
        text = urllib.request.urlopen(url).read().decode("utf-8")
    rows = [line.strip() for line in text.split("\n") if line.strip()]
    return pd.DataFrame({"comment": rows})

# ----- Choose your data -----------------------------------------------------------
# EXPECTED PATH -- bring your own: load the cleaned CSV you made in the Day 10 workshop.
#     df = pd.read_csv("YOUR_FILE.csv")
#     text_column = "the_text_column"      # e.g. "comment"
#
# FALLBACK (runs out of the box) -- the course corpus on the TX Ten Commandments law:
df = load_course_comments()
text_column = "comment"
# ---------------------------------------------------------------------------------

print(f"{len(df)} rows loaded. Text column: '{text_column}'.")
df.head()

## Part A — From text to a sentiment score (5 exercises)

**A1 — Meet your data as a table.**

In [ ]:
# A1 — solution
print(df.shape)          # (123, 1)
print(len(df), "comments")
df.head()

> **Teaching note:** Goal: read a DataFrame's shape; each row = one comment (bridge from HW2's
> single blob). The corpus is **123 rows**; a handful are short wrap-fragments ("Of Texas", "The")
> left by the CSV→txt cleaning — a realistic data-quality wart worth naming, and a preview of why
> the Day-10 cleaning workshop matters.

**A2 — Clean text, but keep the feeling.**

In [ ]:
# A2 — solution
sample = df[text_column].iloc[0]
print("Original:   ", sample)
print("Lowercased: ", sample.lower())
# Lowercasing + stripping "!" is RIGHT for term frequency (HW2) but WRONG for sentiment:
# "GREAT!!!" and "great" carry different intensity. VADER wants the raw text.

> **Teaching note:** Goal: preprocessing is an interpretive choice — the "right" cleaning depends
> on the question. HW2 wanted word identity (drop case/punctuation); VADER wants intensity (keep them).

**A3 — Score a single comment with VADER.**

In [ ]:
# A3 — solution
text = "This is a great day for religious freedom!"
scores = analyzer.polarity_scores(text)
print(scores)               # {'neg':0.0, 'neu':0.411, 'pos':0.589, 'compound':0.8622}
print(scores["compound"])   # 0.8622 -- the single summary score, -1..+1

> **Teaching note:** Goal: read VADER's dict. `compound` is the one-number summary; `pos/neu/neg`
> are proportions that sum to ~1. Students only need `compound` going forward.

**A4 — An edge case: predict, then run.**

In [ ]:
# A4 — solution
sarcasm = "Oh great, another set of commandments to follow."
print(analyzer.polarity_scores(sarcasm)["compound"])
# +0.625 -- VADER reads "great" as positive and MISSES the sarcasm; a human reads it as negative.

> **Teaching note:** Goal: predict-then-run (like HW1/HW2 A4), here exposing VADER's blind spot —
> sarcasm, irony, context. This is the critical heart of the assignment and the bridge to the
> human-vs-VADER check (B3) and Day 7 (AI Agency). Tools are useful AND fallible.

**A5 — Score every comment at once.**

In [ ]:
# A5 — solution
def get_sentiment(text):
    return analyzer.polarity_scores(text)["compound"]

df["sentiment"] = df[text_column].apply(get_sentiment)
print("mean:", round(df["sentiment"].mean(), 3))   # 0.082
print("min: ", round(df["sentiment"].min(), 3))    # -0.968
print("max: ", round(df["sentiment"].max(), 3))    # 0.92
df.head()

> **Teaching note:** Goal: `.apply()` = HW2's hand-loop, vectorized over the column. The near-zero
> **mean (0.082)** is itself a finding — the crowd is split, not uniformly for or against. Preview of B.

### A note: you didn't build VADER — and that's normal
VADER is a **sentiment lexicon**: thousands of words hand-scored for feeling, plus rules for
"!", ALL CAPS, and negation, compiled and tuned by researchers. Almost nobody writes one from
scratch — you `pip install` it. (Long before AI, you'd still have reached for a library like
this; today you might ask an AI to help use it.) **Borrowing it is expected, not cheating.**

But A4 showed the catch: a borrowed tool carries someone else's assumptions and can be
**confidently wrong** (it missed the sarcasm). So the real skill isn't *building* VADER — it's
**judging whether to trust it** on your text. That's exactly what the human-vs-VADER check in
**B3** is for. *(We pick this thread up on **Day 7**, reading and improving AI-written code.)*

## Part B — Support, opposition, and what counting missed (4 exercises)

**B1 — Turn scores into labels.**

In [ ]:
# B1 — solution
def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    return "neutral"

df["label"] = df["sentiment"].apply(label_sentiment)
print(df["label"].value_counts())
# positive    51
# neutral     38
# negative    34

> **Teaching note:** Goal: thresholds turn a continuum into bins — an authored choice (callback to
> HW1 A3 age bins / Classification Logic). 0.05 is a convention, not a fact; moving it reshapes the
> story. Note positives lead but neutral+negative (72) outweigh them — a divided crowd, not a mandate.

**B2 — Picture the split.**

In [ ]:
# B2 — solution
df["label"].value_counts().plot(kind="bar", title="Sentiment of the comments")
plt.ylabel("number of comments")
plt.show()
# Positive bar slightly tallest, but neutral + negative together are the larger story: a split crowd.

> **Teaching note:** Goal: one-line pandas viz (`df.plot`) reinforcing the code-alongs — not new
> material. Under ungrading, "make a chart and say what it shows" is the whole bar; reward the reading,
> not chart polish.

**B3 — Read the extremes. Do you agree with VADER?**

In [ ]:
# B3 — solution
most_positive = df.loc[df["sentiment"].idxmax()]
most_negative = df.loc[df["sentiment"].idxmin()]
print("MOST POSITIVE (", round(most_positive["sentiment"], 3), "):", most_positive[text_column])
print("MOST NEGATIVE (", round(most_negative["sentiment"], 3), "):", most_negative[text_column])
# Most positive ~0.920; most negative ~-0.968. Students read both and note where VADER's number
# matches their gut and where tone/context complicate it.

> **Teaching note:** Goal: deterministic close-reading of the extremes (`idxmax`/`idxmin` — no random
> `sample()`, so the key is reproducible) plus the human-vs-automated judgment F25 ran on a 5-row
> sample. The assignment's critical core: **reward thoughtful disagreement with VADER, not "right"
> labels.** Great place to surface that the most-negative comment may be messy/ambiguous text.

**B4 — Interpretation (model answer).**

> **Model answer (accept variants; look for ≥2 numbers + a named VADER failure):** The crowd is
> **split, not a mandate** — 51 positive vs. 38 neutral and 34 negative, with a near-zero mean
> (0.082). Sentiment adds what HW2 couldn't: in HW2 *commandments* "won" on raw frequency, but here
> it tops **both** the positive and negative word lists (C1) — the same word was a cheer for some and
> a complaint for others, which a count alone flattened into one "winner." **What VADER still gets
> wrong:** sarcasm and scripture-quoting (A4's "Oh great…" scores +0.625; a "John 3:16" comment may
> score neutral despite a strong stance), so the labels need a human read.

> **Teaching note:** Reward any answer that (a) cites ≥2 concrete numbers and (b) names a real VADER
> limitation. This is the HW2→HW3→HW4 spine: count → stance → integration.

## Part C — Going deeper (2 exercises)

**C1 — Whose words are whose? Frequency *inside* each camp.**

In [ ]:
# C1 — solution
def top_words(text_series, n=8):
    words = []
    for t in text_series:
        words.extend(w for w in split_into_words(t) if w and w not in stopwords)
    return Counter(words).most_common(n)

print("POSITIVE:", top_words(df[df["label"] == "positive"][text_column]))
print("NEGATIVE:", top_words(df[df["label"] == "negative"][text_column]))
# POSITIVE: religion 11, commandments 8, god 7, freedom 6, children 6, want 6, religious 5, good 5
# NEGATIVE: commandments 11, ten 10, law 7, bible 6, god 6, schools 5, court 5, broken 4
# 'commandments' and 'god' top BOTH camps -- HW2's frequency "winners" are shared vocabulary used
# to argue opposite sides. Sentiment is what separates them.

> **Teaching note:** Goal: the synthesis — reuse HW2's term-frequency idea on sentiment-split
> subsets. Payoff: HW2's winning words (*commandments*, *god*) appear on **both** sides, so frequency
> alone conflated support and opposition; sentiment disentangles them. The whole HW2→HW3 arc in one
> cell, and the setup for HW4's integration.

**C2 — On your own data, or push further (model).**

In [ ]:
# C2 — solution (one valid version of the "constitution" comparison)
mentions = df[df[text_column].str.contains("constitution", case=False)]
others = df[~df[text_column].str.contains("constitution", case=False)]
print("mentions 'constitution':", len(mentions), "| mean sentiment:", round(mentions["sentiment"].mean(), 3))
print("does not:               ", len(others), "| mean sentiment:", round(others["sentiment"].mean(), 3))

> **Teaching note:** Goal: open exploration; any defensible version earns full credit under
> ungrading. For BYO submissions, evaluate the reflection on *their* corpus + a named VADER doubt,
> not a fixed number. The "constitution" cut is a nice callback to HW2's pairing.